In [3]:
# ============================================================
# CUSTOMER TRANSACTION PREDICTION
# PRCP-1003 | Binary Classification Pipeline
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.simplefilter('ignore')
import os # Import the os module

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve, precision_recall_curve,
                              average_precision_score)
from sklearn.utils import resample
import lightgbm as lgb

# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = 'train(1).csv'   # <-- Change to your CSV path
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET_COL = 'target'
ID_COL = 'ID_code'
OUTPUT_DIR = 'outputs' # Define an output directory

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# STEP 1: LOAD DATA
# ============================================================
print("=" * 60)
print("  CUSTOMER TRANSACTION PREDICTION")
print("=" * 60)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"    Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

# ============================================================
# STEP 2: EXPLORATORY DATA ANALYSIS
# ============================================================
print("\n[2] Exploratory Data Analysis")
print("-" * 40)

# Basic info
print(f"    Missing values : {df.isnull().sum().sum()}")
print(f"    Duplicate rows : {df.duplicated().sum()}")
print(f"\n    Target Distribution:")
target_counts = df[TARGET_COL].value_counts()
for val, cnt in target_counts.items():
    label = "Will NOT transact (0)" if val == 0 else "Will transact     (1)"
    pct = cnt / len(df) * 100
    print(f"      {label} → {cnt:,} ({pct:.1f}%)")

imbalance_ratio = target_counts[0] / target_counts[1]
print(f"\n    Imbalance ratio  : {imbalance_ratio:.1f}:1")

# Feature stats
feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
print(f"    Feature columns  : {len(feature_cols)}")
print(f"\n    Sample statistics (first 5 features):")
print(df[feature_cols[:5]].describe().round(3).to_string())

# ============================================================
# STEP 3: EDA VISUALIZATIONS
# ============================================================
print("\n[3] Generating EDA visualizations...")

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Customer Transaction Prediction — EDA Report', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- Plot 1: Target Distribution (Pie) ---
ax1 = fig.add_subplot(gs[0, 0])
colors = ['#4C72B0', '#DD8452']
wedges, texts, autotexts = ax1.pie(
    target_counts.values, labels=['No Transaction\n(0)', 'Transaction\n(1)'],
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops=dict(width=0.6))
for at in autotexts:
    at.set_fontsize(10)
ax1.set_title('Target Distribution', fontweight='bold')

# --- Plot 2: Target Count Bar ---
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(['No Transaction (0)', 'Transaction (1)'], target_counts.values,
               color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, target_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{val:,}', ha='center', fontsize=10, fontweight='bold')
ax2.set_title('Class Count', fontweight='bold')
ax2.set_ylabel('Count')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Plot 3: Mean feature value by class (top 10 features) ---
ax3 = fig.add_subplot(gs[0, 2])
class_means = df.groupby(TARGET_COL)[feature_cols[:20]].mean()
diff = (class_means.loc[1] - class_means.loc[0]).abs().sort_values(ascending=False)
top10 = diff.head(10)
ax3.barh(range(len(top10)), top10.values, color='#55A868')
ax3.set_yticks(range(len(top10)))
ax3.set_yticklabels([f'var_{i}' if 'var_' not in n else n for n, i in zip(top10.index, range(10))], fontsize=8)
ax3.set_title('Top 10 Features\n(Mean Difference by Class)', fontweight='bold')
ax3.set_xlabel('|Mean Diff|')

# --- Plot 4: Distribution of var_0 by class ---
ax4 = fig.add_subplot(gs[1, 0])
for cls, color, label in [(0, '#4C72B0', 'No Trans (0)'), (1, '#DD8452', 'Trans (1)')]:
    vals = df[df[TARGET_COL] == cls][feature_cols[0]]
    ax4.hist(vals, bins=50, alpha=0.6, color=color, label=label, density=True)
ax4.set_title(f'Distribution of {feature_cols[0]}', fontweight='bold')
ax4.set_xlabel(feature_cols[0])
ax4.set_ylabel('Density')
ax4.legend(fontsize=8)

# --- Plot 5: Distribution of var_1 by class ---
ax5 = fig.add_subplot(gs[1, 1])
for cls, color, label in [(0, '#4C72B0', 'No Trans (0)'), (1, '#DD8452', 'Trans (1)')]:
    vals = df[df[TARGET_COL] == cls][feature_cols[1]]
    ax5.hist(vals, bins=50, alpha=0.6, color=color, label=label, density=True)
ax5.set_title(f'Distribution of {feature_cols[1]}', fontweight='bold')
ax5.set_xlabel(feature_cols[1])
ax5.set_ylabel('Density')
ax5.legend(fontsize=8)

# --- Plot 6: Correlation heatmap (first 10 features) ---
ax6 = fig.add_subplot(gs[1, 2])
corr = df[feature_cols[:10]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax6, cmap='coolwarm', center=0,
            annot=False, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax6.set_title('Feature Correlation\n(First 10 vars)', fontweight='bold')
ax6.tick_params(labelsize=7)

# --- Plot 7: Boxplot of top 5 features ---
ax7 = fig.add_subplot(gs[2, :])
top5_feats = top10.index[:5].tolist()
plot_data = []
labels = []
colors_box = []
for feat in top5_feats:
    for cls in [0, 1]:
        plot_data.append(df[df[TARGET_COL] == cls][feat].values)
        labels.append(f'{feat}\n({"No" if cls==0 else "Yes"})')
        colors_box.append('#4C72B0' if cls == 0 else '#DD8452')

bp = ax7.boxplot(plot_data, patch_artist=True, notch=False,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax7.set_xticklabels(labels, fontsize=8)
ax7.set_title('Top 5 Discriminative Features — Distribution by Class', fontweight='bold')
ax7.set_ylabel('Value')

plt.savefig(os.path.join(OUTPUT_DIR, 'eda_report.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f"    Saved: {os.path.join(OUTPUT_DIR, 'eda_report.png')}")

# ============================================================
# STEP 4: DATA PREPROCESSING & BALANCING
# ============================================================
print("\n[4] Preprocessing & Balancing...")

df_majority = df[df[TARGET_COL] == 0]
df_minority = df[df[TARGET_COL] == 1]
n_majority = len(df_majority)

# Upsample minority to match majority
df_minority_upsampled = resample(df_minority, replace=True,
                                  n_samples=n_majority,
                                  random_state=RANDOM_STATE)
df_balanced = pd.concat([df_majority, df_minority_upsampled]).sample(frac=1, random_state=RANDOM_STATE)
df_balanced.dropna(inplace=True)

print(f"    Before balancing: {target_counts[0]:,} class-0 | {target_counts[1]:,} class-1")
print(f"    After balancing : {len(df_balanced[df_balanced[TARGET_COL]==0]):,} class-0 | {len(df_balanced[df_balanced[TARGET_COL]==1]):,} class-1")

X = df_balanced[feature_cols]
y = df_balanced[TARGET_COL]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"    Train size: {len(X_train):,} | Test size: {len(X_test):,}")

# ============================================================
# STEP 5: MODEL TRAINING
# ============================================================
print("\n[5] Training Models...")
print("-" * 40)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   n_jobs=-1, random_state=RANDOM_STATE),
    "LightGBM":            lgb.LGBMClassifier(n_estimators=300, max_depth=8,
                                               learning_rate=0.05, num_leaves=31,
                                               n_jobs=-1, random_state=RANDOM_STATE,
                                               verbose=-1)
}

results = {}
for name, model in models.items():
    print(f"\n    Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    acc   = accuracy_score(y_test, preds)
    roc   = roc_auc_score(y_test, proba)
    ap    = average_precision_score(y_test, proba)
    cm    = confusion_matrix(y_test, preds)

    results[name] = {
        'model': model, 'preds': preds, 'proba': proba,
        'accuracy': acc, 'roc_auc': roc, 'avg_precision': ap, 'cm': cm
    }
    print(f"      Accuracy      : {acc:.4f}")
    print(f"      ROC-AUC       : {roc:.4f}")
    print(f"      Avg Precision : {ap:.4f}")

# ============================================================
# STEP 6: DETAILED EVALUATION OF BEST MODEL
# ============================================================
best_name = max(results, key=lambda n: results[n]['roc_auc'])
best = results[best_name]

print("\n" + "=" * 60)
print(f"  BEST MODEL: {best_name}")
print("=" * 60)
print(f"\n  Accuracy      : {best['accuracy']:.4f}")
print(f"  ROC-AUC       : {best['roc_auc']:.4f}")
print(f"  Avg Precision : {best['avg_precision']:.4f}")
print(f"\n  Classification Report:\n")
print(classification_report(y_test, best['preds'],
      target_names=['No Transaction', 'Will Transact']))

# ============================================================
# STEP 7: MODEL COMPARISON VISUALIZATIONS
# ============================================================
print("\n[7] Generating model evaluation plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model Evaluation Report — Customer Transaction Prediction',
             fontsize=15, fontweight='bold')

colors_m = {'Logistic Regression': '#4C72B0', 'Random Forest': '#55A868', 'LightGBM': '#DD8452'}

# --- Plot 1: Accuracy Comparison ---
ax = axes[0, 0]
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
bars = ax.bar(names, accs, color=[colors_m[n] for n in names], edgecolor='white')
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_title('Accuracy Comparison', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.tick_params(axis='x', labelsize=9)

# --- Plot 2: ROC-AUC Comparison ---
ax = axes[0, 1]
rocs = [results[n]['roc_auc'] for n in names]
bars = ax.bar(names, rocs, color=[colors_m[n] for n in names], edgecolor='white')
for bar, val in zip(bars, rocs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_title('ROC-AUC Comparison', fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
ax.tick_params(axis='x', labelsize=9)

# --- Plot 3: ROC Curves ---
ax = axes[0, 2]
for name in names:
    fpr, tpr, _ = roc_curve(y_test, results[name]['proba'])
    ax.plot(fpr, tpr, color=colors_m[name], lw=2,
            label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Plot 4: Confusion Matrix (Best Model) ---
ax = axes[1, 0]
cm = best['cm']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Trans', 'Trans'], yticklabels=['No Trans', 'Trans'],
            linewidths=1, cbar=False, annot_kws={'size': 13})
ax.set_title(f'Confusion Matrix\n({best_name})', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

# --- Plot 5: Precision-Recall Curves ---
ax = axes[1, 1]
for name in names:
    prec, rec, _ = precision_recall_curve(y_test, results[name]['proba'])
    ap = results[name]['avg_precision']
    ax.plot(rec, prec, color=colors_m[name], lw=2,
            label=f"{name} (AP={ap:.3f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Plot 6: Feature Importance (LightGBM) ---
ax = axes[1, 2]
lgb_model = results['LightGBM']['model']
fi = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
ax.barh(range(len(fi)), fi.values[::-1], color='#DD8452')
ax.set_yticks(range(len(fi)))
ax.set_yticklabels(fi.index[::-1], fontsize=8)
ax.set_title('Top 15 Feature Importances\n(LightGBM)', fontweight='bold')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_evaluation.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f"    Saved: {os.path.join(OUTPUT_DIR, 'model_evaluation.png')}")

# ============================================================
# STEP 8: SUMMARY REPORT
# ============================================================
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"\n  {'Model':<25} {'Accuracy':>10} {'ROC-AUC':>10} {'Avg Prec':>10}")
print(f"  {'-'*55}")
for name in names:
    marker = " ← BEST" if name == best_name else ""
    print(f"  {name:<25} {results[name]['accuracy']:>10.4f} "
          f"{results[name]['roc_auc']:>10.4f} "
          f"{results[name]['avg_precision']:>10.4f}{marker}")

print(f"""
  RECOMMENDATION
  ──────────────
  ✔  Best Model     : {best_name}
  ✔  ROC-AUC        : {best['roc_auc']:.4f}
  ✔  Accuracy       : {best['accuracy']:.4f}

  INSIGHTS
  ────────
  • Dataset had a severe class imbalance (~10:1) — fixed via upsampling.
  • LightGBM outperforms others due to gradient boosting on weak learners.
  • StandardScaler improves Logistic Regression convergence significantly.
  • Top features driving prediction are identified in the importance plot.

  OUTPUT FILES
  ────────────
  → {os.path.join(OUTPUT_DIR, 'eda_report.png')}        (EDA visualizations)
  → {os.path.join(OUTPUT_DIR, 'model_evaluation.png')}  (Model comparison & metrics)

  To use the best model on new data:
    predictions = results['{best_name}']['model'].predict(X_new_scaled)
    probabilities = results['{best_name}']['model'].predict_proba(X_new_scaled)[:, 1]
""")
print("=" * 60)
print("  Done!")
print("=" * 60)


  CUSTOMER TRANSACTION PREDICTION

[1] Loading dataset...
    Shape: 4,169 rows × 202 columns

[2] Exploratory Data Analysis
----------------------------------------
    Missing values : 84
    Duplicate rows : 0

    Target Distribution:
      Will NOT transact (0) → 3,751 (90.0%)
      Will transact     (1) → 418 (10.0%)

    Imbalance ratio  : 9.0:1
    Feature columns  : 200

    Sample statistics (first 5 features):
          var_0     var_1     var_2     var_3     var_4
count  4169.000  4169.000  4169.000  4169.000  4169.000
mean     10.732    -1.642    10.686     6.820    11.086
std       3.012     4.118     2.654     2.077     1.620
min       1.335   -13.423     3.678     1.154     6.140
25%       8.530    -4.765     8.658     5.200     9.865
50%      10.584    -1.629    10.535     6.847    11.101
75%      12.753     1.285    12.466     8.391    12.276
max      19.289     8.416    18.348    12.674    15.111

[3] Generating EDA visualizations...
    Saved: outputs/eda_report.png